# Machine Learning Models for Predicting Early Malaria Outbreaks in Young Children
Six models trained on the 2019 Ghana MIS and tested on the 2022 Ghana DHS.

## Setup

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from scipy.stats import loguniform, randint, uniform
import hashlib
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from google.colab import drive
import sklearn, xgboost, scipy

warnings.filterwarnings("ignore")
SEED = 2026
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scipy", scipy.__version__)

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/malaria_u5")
OUT = PROJECT / "outputs"
OUT.mkdir(exist_ok=True)

## Data

In [ ]:
DATA = PROJECT / "malaria_u5_dataset.csv"
EXPECTED_MD5 = "0c8e6c3138c378cec514657158045ebb"
md5 = hashlib.md5(DATA.read_bytes()).hexdigest()
print("Dataset MD5:", md5, "(matches the thesis dataset)" if md5 == EXPECTED_MD5 else "WARNING: not the thesis dataset")
d = pd.read_csv(DATA)

def wprev(g, col):
    g = g.dropna(subset=[col])
    return round(100 * np.average(g[col], weights=g.weight), 1)

d.groupby("survey_year").apply(lambda g: pd.Series({
    "children": len(g), "rdt_positive": g.malaria_rdt.sum(), "clusters": g.cluster.nunique(),
    "weighted_rdt_%": wprev(g, "malaria_rdt"), "weighted_microscopy_%": wprev(g, "malaria_microscopy")})
).astype({"children": int, "rdt_positive": int, "clusters": int})

## Predictors

In [ ]:
CHILD_HH = ["age_months", "wealth_quintile", "household_size", "children_under5", "nets_per_person",
            "housing_finished_count", "sanitation_score", "female", "urban", "slept_under_itn",
            "electricity", "sprayed_irs_12m", "floor_finished", "wall_finished", "roof_finished"]
ENV = ["altitude_m", "gc_aridity", "gc_mean_temperature", "gc_maximum_temperature", "gc_minimum_temperature",
       "gc_rainfall", "gc_enhanced_vegetation_index", "gc_pet", "gc_wet_days", "gc_land_surface_temperature",
       "gc_diurnal_temperature_range", "gc_un_population_density", "gc_global_human_footprint"]
WEATHER = ["cl_rain_lag1", "cl_rain_lag2", "cl_rain_lag3", "cl_rain_3m",
           "cl_temp_lag1", "cl_temp_3m", "cl_rh_lag1", "cl_rh_3m"]
WEATHER_REG = [c.replace("cl_", "reg_") for c in WEATHER]
CAT = ["mother_education", "water_jmp", "region10"]
NUM = CHILD_HH + ENV + WEATHER

LABELS = {
    "age_months": "Child age (months)", "wealth_quintile": "Wealth quintile", "household_size": "Household size",
    "children_under5": "Children under 5", "nets_per_person": "Bed nets per person",
    "housing_finished_count": "Finished housing materials", "sanitation_score": "Sanitation (JMP)",
    "female": "Female", "urban": "Urban", "slept_under_itn": "Slept under ITN", "electricity": "Electricity",
    "sprayed_irs_12m": "Indoor residual spraying", "floor_finished": "Finished floor",
    "wall_finished": "Finished walls", "roof_finished": "Finished roof", "mother_education": "Mother's education",
    "water_jmp": "Drinking water (JMP)", "region10": "Region", "altitude_m": "Altitude",
    "gc_aridity": "Aridity", "gc_mean_temperature": "Mean temperature", "gc_maximum_temperature": "Max temperature",
    "gc_minimum_temperature": "Min temperature", "gc_rainfall": "Annual rainfall",
    "gc_enhanced_vegetation_index": "Vegetation (EVI)", "gc_pet": "Evapotranspiration", "gc_wet_days": "Wet days",
    "gc_land_surface_temperature": "Land surface temperature", "gc_diurnal_temperature_range": "Diurnal temperature range",
    "gc_un_population_density": "Population density", "gc_global_human_footprint": "Human footprint",
    "cl_rain_lag1": "Rainfall, 1 month before",
    "cl_rain_lag2": "Rainfall, 2 months before", "cl_rain_lag3": "Rainfall, 3 months before",
    "cl_rain_3m": "Rainfall, 3 months total", "cl_temp_lag1": "Temperature, 1 month before",
    "cl_temp_3m": "Temperature, 3-month mean", "cl_rh_lag1": "Humidity, 1 month before",
    "cl_rh_3m": "Humidity, 3-month mean"}
print(len(NUM) + len(CAT), "predictors")

In [ ]:
miss = pd.DataFrame({f"{y} children": d[d.survey_year == y][NUM + CAT].isna().sum() for y in (2019, 2022)})
miss["2019 clusters"] = [d[(d.survey_year == 2019) & d[c].isna()].cluster.nunique() for c in miss.index]
miss["2022 clusters"] = [d[(d.survey_year == 2022) & d[c].isna()].cluster.nunique() for c in miss.index]
print("Children with at least one missing predictor:",
      {y: int(d[d.survey_year == y][NUM + CAT].isna().any(axis=1).sum()) for y in (2019, 2022)})
miss[miss.iloc[:, :2].sum(axis=1) > 0].rename(index=LABELS)

## Functions

In [ ]:
def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"threshold": round(float(thr), 3), "accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred),
            "specificity": tn / (tn + fp), "f1": f1_score(y, pred, zero_division=0),
            "TP": tp, "FP": fp, "TN": tn, "FN": fn}

def threshold_for_sensitivity(y, p, target=0.80):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr >= target)])

def cluster_resamples(clusters, strata, n):
    # whole clusters resampled within each sampling stratum, as in the survey design
    rng = np.random.default_rng(SEED)
    df = pd.DataFrame({"g": np.asarray(clusters), "s": np.asarray(strata)})
    rows = {g: np.where(df.g.values == g)[0] for g in df.g.unique()}
    by_stratum = [df[df.s == s].g.unique() for s in df.s.unique()]
    for _ in range(n):
        yield np.concatenate([rows[g] for cl in by_stratum for g in rng.choice(cl, len(cl))])

def bootstrap_auc_ci(y, p, clusters, strata, n=1000):
    y, p = np.asarray(y), np.asarray(p)
    aucs = [roc_auc_score(y[s], p[s]) for s in cluster_resamples(clusters, strata, n) if len(np.unique(y[s])) == 2]
    return np.percentile(aucs, [2.5, 97.5])

def calibration(y, p, w=None):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    fw = None if w is None else np.asarray(w)
    slope = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial(), freq_weights=fw).fit().params[1]
    intercept = sm.GLM(y, np.ones_like(lp), family=sm.families.Binomial(), offset=lp, freq_weights=fw).fit().params[0]
    return intercept, slope

class RegionMedianImputer(BaseEstimator, TransformerMixin):
    """Fills missing numeric values with the training-data median of the same region (overall median if none)."""
    def __init__(self, num, group="region10"):
        self.num, self.group = num, group
    def fit(self, X, y=None):
        self.region_ = X.groupby(self.group)[self.num].median()
        self.overall_ = X[self.num].median()
        return self
    def transform(self, X):
        X = X.copy()
        fill = self.region_.reindex(X[self.group].values).set_axis(X.index).fillna(self.overall_)
        X[self.num] = X[self.num].fillna(fill)
        return X

def delong(y, p1, p2):
    def midrank(x):
        j = np.argsort(x); z = x[j]; n = len(x); t = np.zeros(n); i = 0
        while i < n:
            k = i
            while k < n and z[k] == z[i]:
                k += 1
            t[i:k] = 0.5 * (i + k - 1) + 1
            i = k
        r = np.empty(n); r[j] = t
        return r
    order = np.argsort(-np.asarray(y)); preds = np.vstack([p1, p2])[:, order]
    m = int(np.sum(y)); n = len(y) - m
    tx = np.array([midrank(p[:m]) for p in preds]); ty = np.array([midrank(p[m:]) for p in preds])
    tz = np.array([midrank(p) for p in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1) / (2 * n)
    s = np.cov((tz[:, :m] - tx) / n) / m + np.cov(1 - (tz[:, m:] - ty) / m) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(s[0, 0] + s[1, 1] - 2 * s[0, 1])
    return aucs, 2 * stats.norm.sf(abs(z))

def plot_confusion_heatmap(r, model_name, filename, labels=("Negative", "Positive")):
    cm = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
                xticklabels=list(labels), yticklabels=list(labels))
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.savefig(OUT / filename, dpi=200)
    plt.show()

## Step 1. Split
2019 for development, 2022 for the final test.

In [ ]:
dev = d[d.survey_year == 2019].reset_index(drop=True)
test = d[d.survey_year == 2022].reset_index(drop=True)
Xd, yd = dev[NUM + CAT], dev.malaria_rdt.values
Xt, yt = test[NUM + CAT], test.malaria_rdt.values
print(f"2019: {len(dev)} children, {yd.sum()} positive | 2022: {len(test)} children, {yt.sum()} positive")

## Step 2. Train and tune six models
Randomised search with 5-fold cross-validation on 2019. Folds are grouped by cluster. Missing numeric values are filled with the regional median of the training data.

In [ ]:
def preprocessor(num, cat, scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale else [])
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)])

def models(num, cat):
    pipe = lambda clf, scale=False: Pipeline([("region_impute", RegionMedianImputer(num)),
                                              ("prep", preprocessor(num, cat, scale)), ("clf", clf)])
    return {
        "Decision Tree": (pipe(DecisionTreeClassifier(random_state=SEED)),
            {"clf__max_depth": [2, 3, 4, 5, 6, 8, 10], "clf__min_samples_leaf": randint(10, 150),
             "clf__criterion": ["gini", "entropy"]}),
        "Logistic Regression": (pipe(LogisticRegression(max_iter=5000), scale=True),
            {"clf__C": loguniform(1e-3, 10)}),
        "Random Forest": (pipe(RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED)),
            {"clf__max_depth": [4, 6, 8, 12, None], "clf__min_samples_leaf": randint(5, 60),
             "clf__max_features": ["sqrt", 0.3, 0.5]}),
        "KNN": (pipe(KNeighborsClassifier(), scale=True),
            {"clf__n_neighbors": randint(15, 301), "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]}),
        "Gradient Boosting": (pipe(GradientBoostingClassifier(random_state=SEED)),
            {"clf__n_estimators": randint(100, 500), "clf__learning_rate": loguniform(0.01, 0.2),
             "clf__max_depth": randint(2, 5), "clf__subsample": uniform(0.6, 0.4),
             "clf__min_samples_leaf": randint(10, 80)}),
        "XGBoost": (pipe(XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=SEED, n_jobs=-1)),
            {"clf__n_estimators": randint(100, 600), "clf__max_depth": randint(2, 6),
             "clf__learning_rate": loguniform(0.01, 0.2), "clf__subsample": uniform(0.6, 0.4),
             "clf__colsample_bytree": uniform(0.5, 0.5), "clf__min_child_weight": randint(1, 20),
             "clf__reg_lambda": loguniform(0.1, 20)}),
    }

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(Xd, yd, dev.cluster_uid))

fitted, params, oof, cv_rows = {}, {}, {}, []
for name, (pipe, grid) in models(NUM, CAT).items():
    search = RandomizedSearchCV(pipe, grid, n_iter=15 if name == "Logistic Regression" else 25,
                                scoring="roc_auc", cv=folds, random_state=SEED, n_jobs=-1).fit(Xd, yd)
    fitted[name] = search.best_estimator_
    params[name] = {k[5:]: v for k, v in search.best_params_.items()}
    oof[name] = np.zeros(len(yd))
    fold_scores = []
    for tr, va in folds:
        p = clone(search.best_estimator_).fit(Xd.iloc[tr], yd[tr]).predict_proba(Xd.iloc[va])[:, 1]
        oof[name][va] = p
        fold_scores.append({**metrics_at(yd[va], p, 0.5), "roc_auc": roc_auc_score(yd[va], p)})
    fs = pd.DataFrame(fold_scores)
    cv_rows.append({"model": name, "train_auc": roc_auc_score(yd, fitted[name].predict_proba(Xd)[:, 1]),
                    **{f"cv_{m}": fs[m].mean() for m in ["roc_auc", "accuracy", "precision", "recall", "f1"]},
                    "cv_roc_auc_sd": fs.roc_auc.std()})
    print(f"{name:20s} CV AUC {fs.roc_auc.mean():.3f}")

cv_table = pd.DataFrame(cv_rows)
cv_table.round(3)

In [ ]:
pd.DataFrame(params).T

## Step 3. Select the best model
Highest cross-validated ROC-AUC on 2019.

In [ ]:
BEST = cv_table.sort_values("cv_roc_auc", ascending=False).model.iloc[0]
print("Selected:", BEST)

## Step 4. Set the threshold
Threshold reaching 80% sensitivity in the cross-validated 2019 predictions.

In [ ]:
THRESHOLD = threshold_for_sensitivity(yd, oof[BEST])
print(f"Threshold: {THRESHOLD:.3f}")
pd.DataFrame([metrics_at(yd, oof[BEST], t) for t in [0.1, 0.2, THRESHOLD, 0.3, 0.4, 0.5]]).round(3)

## Step 5. Final test on 2022

In [ ]:
p_best = fitted[BEST].predict_proba(Xt)[:, 1]
final = metrics_at(yt, p_best, THRESHOLD)
lo, hi = bootstrap_auc_ci(yt, p_best, test.cluster_uid, test.strata)
cal_int, cal_slope = calibration(yt, p_best)
final_table = pd.Series({"model": BEST, "threshold": THRESHOLD, "roc_auc": roc_auc_score(yt, p_best),
                         "auc_95ci": f"{lo:.3f}-{hi:.3f}", **{k: final[k] for k in
                         ["accuracy", "precision", "recall", "specificity", "f1", "TP", "FP", "TN", "FN"]},
                         "pr_auc": average_precision_score(yt, p_best), "brier": brier_score_loss(yt, p_best),
                         "calibration_intercept": cal_int, "calibration_slope": cal_slope})
final_table

In [ ]:
# survey-weighted performance of the selected model (sensitivity analysis)
w_cal = calibration(yt, p_best, test.weight / test.weight.mean())
weighted = pd.Series({"weighted_roc_auc": roc_auc_score(yt, p_best, sample_weight=test.weight),
                      "weighted_observed": np.average(yt, weights=test.weight),
                      "weighted_mean_predicted": np.average(p_best, weights=test.weight),
                      "weighted_calibration_intercept": w_cal[0], "weighted_calibration_slope": w_cal[1]})
weighted.round(3)

In [ ]:
plot_confusion_heatmap(final, BEST, "fig1a_confusion_matrix_selected_model.png")

fpr, tpr, _ = roc_curve(yt, p_best)
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color="tab:green", label=f"{BEST} (AUC = {roc_auc_score(yt, p_best):.2f})")
plt.scatter(1 - final["specificity"], final["recall"], color="black", zorder=3, label=f"Threshold = {THRESHOLD:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.title(f"ROC Curve - {BEST} (Test Set 2022)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.grid(True)
plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(OUT / "fig1b_roc_selected_model.png", dpi=200); plt.show()

## Step 6. Compare all six models on 2022
Each model uses its own threshold set on 2019.

In [ ]:
preds, rows = {}, []
for name, mdl in fitted.items():
    preds[name] = p = mdl.predict_proba(Xt)[:, 1]
    lo_, hi_ = bootstrap_auc_ci(yt, p, test.cluster_uid, test.strata)
    ci_, cs_ = calibration(yt, p)
    common = {"model": name, "roc_auc": roc_auc_score(yt, p), "auc_ci_low": lo_, "auc_ci_high": hi_,
              "pr_auc": average_precision_score(yt, p), "brier": brier_score_loss(yt, p),
              "calibration_intercept": ci_, "calibration_slope": cs_}
    rows.append({**common, "rule": "2019 threshold", **metrics_at(yt, p, threshold_for_sensitivity(yd, oof[name]))})
    rows.append({**common, "rule": "0.50", **metrics_at(yt, p, 0.5)})
test_table = pd.DataFrame(rows)
cols = ["model", "threshold", "roc_auc", "auc_ci_low", "auc_ci_high", "accuracy", "precision",
        "recall", "specificity", "f1", "TP", "FP", "TN", "FN"]
test_table[test_table.rule == "2019 threshold"][cols].round(3)

In [ ]:
test_table[test_table.rule == '0.50'][cols].round(3)

In [ ]:
auc_table = cv_table[["model", "train_auc", "cv_roc_auc"]].assign(
    test_auc=lambda t: t.model.map(lambda m: roc_auc_score(yt, preds[m])))
auc_table.round(3)

In [ ]:
delong_table = pd.DataFrame([
    {"comparison": f"{BEST} vs {m}", "auc_best": a[0], "auc_other": a[1], "p_value": pv}
    for m in preds if m != BEST for a, pv in [delong(yt, preds[BEST], preds[m])]])
delong_table["p_bonferroni"] = (delong_table.p_value * len(delong_table)).clip(upper=1)
delong_table.round(4)

In [ ]:
baseline_table = pd.DataFrame([
    {"comparison": f"{m} vs Logistic Regression", "auc_model": a[0], "auc_lr": a[1], "p_value": pv}
    for m in preds if m != "Logistic Regression"
    for a, pv in [delong(yt, preds[m], preds["Logistic Regression"])]])
baseline_table["p_bonferroni"] = (baseline_table.p_value * len(baseline_table)).clip(upper=1)
baseline_table.round(4)

In [ ]:
def paired_cluster_bootstrap(p1, p2, n=1000):
    # AUC difference p1 - p2, whole clusters resampled within strata (same resamples for both models)
    diffs = [roc_auc_score(yt[s], p1[s]) - roc_auc_score(yt[s], p2[s])
             for s in cluster_resamples(test.cluster_uid, test.strata, n) if len(np.unique(yt[s])) == 2]
    return np.percentile(diffs, [2.5, 97.5])

cluster_table = pd.DataFrame([
    {"comparison": f"{m} vs {ref}", "auc_difference": roc_auc_score(yt, preds[m]) - roc_auc_score(yt, preds[ref]),
     "ci_low": lo_, "ci_high": hi_}
    for ref in ["Logistic Regression", BEST] for m in preds if m != ref
    for lo_, hi_ in [paired_cluster_bootstrap(preds[m], preds[ref])]])
cluster_table.round(4)

In [ ]:
plt.figure(figsize=(10, 7))
for name, p in preds.items():
    fpr, tpr, _ = roc_curve(yt, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(yt, p):.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.title("ROC Curves for All Models (Test Set 2022)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.grid(True)
plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(OUT / "fig2_roc_all_models.png", dpi=200); plt.show()

In [ ]:
for _, r in test_table[test_table.rule == "2019 threshold"].iterrows():
    plot_confusion_heatmap(r, r.model, f"fig3_confusion_matrix_{r.model.lower().replace(' ', '_')}.png")

In [ ]:
t = test_table[test_table.rule == "2019 threshold"].reset_index(drop=True)
metrics = {"accuracy": "Test Accuracy", "precision": "Test Precision", "recall": "Test Recall",
           "f1": "Test F1-score", "roc_auc": "Test ROC-AUC"}
x = np.arange(len(t)); bar_width = 0.16
plt.figure(figsize=(12, 6))
for i, (col, label) in enumerate(metrics.items()):
    plt.bar(x + (i - 2) * bar_width, t[col], width=bar_width, label=label)
plt.xticks(x, t.model, rotation=15)
plt.ylabel("Score")
plt.ylim(0.0, 1.05)
plt.title("Evaluation Metrics by Model (Test Set 2022)")
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout(); plt.savefig(OUT / "fig4_metrics.png", dpi=200); plt.show()

## Step 7. Calibration and feature importance

In [ ]:
plt.figure(figsize=(10, 7))
for name, p in preds.items():
    obs, pred = calibration_curve(yt, p, n_bins=10, strategy="quantile")
    plt.plot(pred, obs, marker="o", label=name)
plt.plot([0, 0.7], [0, 0.7], linestyle="--", color="gray", label="Perfect calibration")
plt.title("Calibration Curves for All Models (Test Set 2022)")
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Proportion Positive")
plt.grid(True)
plt.legend()
plt.tight_layout(); plt.savefig(OUT / "fig5_calibration.png", dpi=200); plt.show()

In [ ]:
print(f"Mean predicted risk 2022 ({BEST}): {p_best.mean():.3f} | observed positivity: {yt.mean():.3f}")
age_band = pd.cut(d.age_months, [5, 11, 23, 35, 47, 59], labels=["6-11", "12-23", "24-35", "36-47", "48-59"])
wmean = lambda g: np.average(g.malaria_rdt, weights=g.weight)
positivity = pd.concat({by: d.assign(age_band=age_band).groupby(["survey_year", col], observed=True).apply(wmean).unstack(0)
                        for by, col in [("age (months)", "age_band"), ("wealth quintile", "wealth_quintile"),
                                        ("urban (1 = urban)", "urban"), ("electricity (1 = yes)", "electricity")]})
positivity.round(3)  # survey-weighted RDT positivity, by survey

In [ ]:
rf = fitted["Random Forest"]
raw = pd.Series(rf.named_steps["clf"].feature_importances_, index=rf.named_steps["prep"].get_feature_names_out())
raw.index = [next((c for c in CAT if n.split("__")[1].startswith(c + "_")), n.split("__")[1]) for n in raw.index]
rf_importance = raw.groupby(level=0).sum().sort_values()
top = rf_importance.tail(15).rename(LABELS)
plt.figure(figsize=(10, 6))
plt.title("Top Feature Importances - Random Forest", fontsize=14)
plt.barh(top.index, top.values, align="center", color="green")
plt.xlabel("Importance Score")
plt.grid(True)
plt.tight_layout(); plt.savefig(OUT / "fig6_rf_importance.png", dpi=200); plt.show()

In [ ]:
pi = permutation_importance(fitted[BEST], Xt, yt, scoring="roc_auc", n_repeats=20, random_state=SEED, n_jobs=-1)
perm = pd.DataFrame({"feature": [LABELS[c] for c in NUM + CAT], "auc_drop": pi.importances_mean,
                     "sd": pi.importances_std}).sort_values("auc_drop")
top = perm.tail(15)
plt.figure(figsize=(10, 6))
plt.title(f"Permutation Feature Importances - {BEST} (Test Set 2022)", fontsize=14)
plt.barh(top.feature, top.auc_drop, xerr=top.sd, align="center", color="green")
plt.xlabel("Drop in ROC-AUC when feature is shuffled")
plt.grid(True)
plt.tight_layout(); plt.savefig(OUT / "fig7_permutation_importance.png", dpi=200); plt.show()

## Step 8. Community high-risk threshold
Mean predicted risk per survey cluster. Threshold fixed on 2019 (out-of-fold predictions), applied to 2022. Main analysis: 2022 communities with at least 8 tested children; high burden = observed positivity in the top quarter of these communities.

In [ ]:
def cluster_risk(frame, p):
    return (frame.assign(pred=p).groupby("cluster_uid")
            .agg(region=("region10", "first"), children=("malaria_rdt", "size"),
                 observed=("malaria_rdt", "mean"), predicted=("pred", "mean")))

c19 = cluster_risk(dev, oof[BEST])
c22 = cluster_risk(test, p_best)
RULES = {"75th percentile": c19.predicted.quantile(0.75),
         "Mean + 2 SD": c19.predicted.mean() + 2 * c19.predicted.std()}
MIN_CHILDREN = 8
c22_all = c22
c22 = c22_all[c22_all.children >= MIN_CHILDREN].copy()
HIGH_BURDEN = c22.observed.quantile(0.75)
c22["high_burden"] = (c22.observed > HIGH_BURDEN).astype(int)
print(f"2019 clusters: {len(c19)} | 2022 clusters: {len(c22_all)} (median {c22_all.children.median():.0f} children) | "
      f"2022 clusters with at least {MIN_CHILDREN} children: {len(c22)} (median {c22.children.median():.0f})")
print(f"Main analysis: {len(c22)} communities with at least {MIN_CHILDREN} children")
print(f"High-burden community in 2022: observed positivity above {HIGH_BURDEN:.3f} (top quartile)")

rows = []
for rule, thr in RULES.items():
    flag = (c22.predicted >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(c22.high_burden, flag, labels=[0, 1]).ravel()
    rows.append({"rule": rule, "threshold": thr, "flagged": int(flag.sum()),
                 "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                 "share_flagged": flag.mean(), "ppv": tp / (tp + fp) if tp + fp else np.nan,
                 "positivity_flagged": c22.observed[flag == 1].mean(),
                 "positivity_not_flagged": c22.observed[flag == 0].mean(),
                 "TP": tp, "FP": fp, "TN": tn, "FN": fn})
community_table = pd.DataFrame(rows)
rho, rho_p = stats.spearmanr(c22.predicted, c22.observed)
print(f"Spearman correlation, predicted vs observed cluster positivity (2022): {rho:.2f} (p = {rho_p:.2g})")
print(f"AUC for identifying high-burden communities: {roc_auc_score(c22.high_burden, c22.predicted):.3f}")
# secondary analysis using 2022 data: intercept-only recalibration, then the same 2019 threshold
logit = lambda q: np.log(q / (1 - q))
p_recal = 1 / (1 + np.exp(-(logit(np.clip(p_best, 1e-6, 1 - 1e-6)) + cal_int)))
c22_recal = cluster_risk(test, p_recal).loc[c22.index]
flag_r = (c22_recal.predicted >= RULES["75th percentile"]).astype(int)
tn, fp, fn, tp = confusion_matrix(c22.high_burden, flag_r, labels=[0, 1]).ravel()
recal_row = {"rule": "75th percentile, after recalibration (uses 2022)", "threshold": RULES["75th percentile"],
             "flagged": int(flag_r.sum()), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
             "share_flagged": flag_r.mean(), "ppv": tp / (tp + fp) if tp + fp else np.nan,
             "positivity_flagged": c22.observed[flag_r == 1].mean(), "positivity_not_flagged": c22.observed[flag_r == 0].mean(),
             "TP": tp, "FP": fp, "TN": tn, "FN": fn}
print(f"Mean predicted risk after recalibration: {p_recal.mean():.3f} (observed {yt.mean():.3f})")
community_table = pd.concat([community_table, pd.DataFrame([recal_row])], ignore_index=True)
community_table.round(3)

In [ ]:
# same 2019 threshold and same high-burden cut-off, applied to communities of different minimum size
robust = []
for k in [1, 5, 8, 10]:
    c = c22_all[c22_all.children >= k]
    hb = (c.observed > HIGH_BURDEN).astype(int)
    flag = (c.predicted >= RULES["75th percentile"]).astype(int)
    tn, fp, fn, tp = confusion_matrix(hb, flag, labels=[0, 1]).ravel()
    robust.append({"min_children": k, "communities": len(c), "high_burden": int(hb.sum()),
                   "share_flagged": flag.mean(), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                   "positivity_flagged": c.observed[flag == 1].mean(),
                   "positivity_not_flagged": c.observed[flag == 0].mean(),
                   "spearman": stats.spearmanr(c.predicted, c.observed)[0], "auc": roc_auc_score(hb, c.predicted)})
robust_table = pd.DataFrame(robust)
robust_table.round(3)

In [ ]:
plot_confusion_heatmap(community_table.iloc[0], "Community High-Risk Threshold",
                       "fig8a_confusion_matrix_community.png", labels=("Not high burden", "High burden"))

thr = RULES["75th percentile"]
plt.figure(figsize=(10, 7))
plt.scatter(c22.predicted, c22.observed, alpha=0.5, color="tab:red")
plt.axvline(thr, linestyle="--", color="black", label=f"Threshold fixed on 2019 = {thr:.2f}")
plt.axhline(HIGH_BURDEN, linestyle=":", color="gray", label=f"High-burden cut-off = {HIGH_BURDEN:.2f}")
plt.title(f"Predicted vs Observed Malaria Positivity, Communities with 8+ Children (Test Set 2022, rho = {rho:.2f})")
plt.xlabel("Mean Predicted Risk")
plt.ylabel("Observed Proportion Positive (RDT)")
plt.grid(True)
plt.legend()
plt.tight_layout(); plt.savefig(OUT / "fig8b_community_risk.png", dpi=200); plt.show()

In [ ]:
region_table = (test.assign(pred=p_best).groupby("region10")
                .agg(children=("malaria_rdt", "size"), observed=("malaria_rdt", "mean"),
                     predicted=("pred", "mean"))
                .join(c22_all.assign(flag=(c22_all.predicted >= thr).astype(int)).groupby("region")
                      .agg(communities=("flag", "size"), flagged=("flag", "sum")))
                .sort_values("predicted", ascending=False))
region_table.round(3)  # communities: all 615 in 2022 (main community analysis uses those with 8+ children)

## Step 9. Performance by subgroup
Selected model on 2022, by residence, region and wealth.

In [ ]:
def subgroup(col, label, min_pos=20):
    rows = []
    for v, g in test.assign(p=p_best).groupby(col):
        pos = int(g.malaria_rdt.sum())
        row = {"group": label, "value": v, "children": len(g), "positive": pos,
               "roc_auc": roc_auc_score(g.malaria_rdt, g.p) if 0 < pos < len(g) else np.nan,
               "ci_low": np.nan, "ci_high": np.nan, "note": ""}
        if pos >= min_pos:
            row["ci_low"], row["ci_high"] = bootstrap_auc_ci(g.malaria_rdt.values, g.p.values, g.cluster_uid, g.strata, n=500)
        else:
            row["note"] = f"fewer than {min_pos} positive children: not interpreted"
        rows.append(row)
    return rows

subgroup_table = pd.DataFrame(subgroup("urban", "Urban (1) / rural (0)") + subgroup("region10", "Region")
                              + subgroup("wealth_quintile", "Wealth quintile"))
subgroup_table.round(3)

## Sensitivity analyses
Selected model with the hyperparameters tuned on the full predictor set: predictor groups, interview season, imputation and outcome. Microscopy uses the RDT-tuned hyperparameters.

In [ ]:
def test_auc(num, train, tst, outcome="malaria_rdt"):
    pipe = models(num, CAT)[BEST][0].set_params(**{f"clf__{k}": v for k, v in params[BEST].items()})
    p = pipe.fit(train[num + CAT], train[outcome].astype(int)).predict_proba(tst[num + CAT])[:, 1]
    y = tst[outcome].astype(int).values
    lo_, hi_ = bootstrap_auc_ci(y, p, tst.cluster_uid, tst.strata, n=500)
    ci_, cs_ = calibration(y, p)
    return p, {"children": len(tst), "positive": int(y.sum()), "test_auc": roc_auc_score(y, p), "ci_low": lo_, "ci_high": hi_,
               "mean_predicted": p.mean(), "observed": y.mean(), "calibration_intercept": ci_, "calibration_slope": cs_}

def paired_diff(p_alt, p_main, n=500):
    d_ = [roc_auc_score(yt[s], p_alt[s]) - roc_auc_score(yt[s], p_main[s])
          for s in cluster_resamples(test.cluster_uid, test.strata, n) if len(np.unique(yt[s])) == 2]
    return roc_auc_score(yt, p_alt) - roc_auc_score(yt, p_main), *np.percentile(d_, [2.5, 97.5])

dev_m, test_m = dev.dropna(subset=["malaria_microscopy"]), test.dropna(subset=["malaria_microscopy"])
oct_nov = test[test.interview_month.isin([10, 11])]
NO_LAG1 = [c for c in NUM if c not in ["cl_rh_lag1", "cl_temp_lag1"]]
runs = {
    "Child, household, education, water and region only": (CHILD_HH, dev, test, "malaria_rdt"),
    "+ cluster environment": (CHILD_HH + ENV, dev, test, "malaria_rdt"),
    "+ cluster environment and regional weather (instead of cluster weather)": (CHILD_HH + ENV + WEATHER_REG, dev, test, "malaria_rdt"),
    "+ cluster environment and cluster weather (main model)": (NUM, dev, test, "malaria_rdt"),
    "Main model without 1-month humidity and temperature": (NO_LAG1, dev, test, "malaria_rdt"),
    "Main model, training children with no missing values": (NUM, dev.dropna(subset=NUM + CAT), test, "malaria_rdt"),
    "Main model, 2022 children interviewed October to November only": (NUM, dev, oct_nov, "malaria_rdt"),
    "Main model, 2022 children interviewed December to January only": (NUM, dev, test[test.interview_month.isin([12, 1])], "malaria_rdt"),
    "Main model, microscopy outcome": (NUM, dev_m, test_m, "malaria_microscopy"),
}
rows, sens_preds = {}, {}
for label, (num, tr_, ts_, outcome) in runs.items():
    sens_preds[label], rows[label] = test_auc(num, tr_, ts_, outcome)
p_main = sens_preds["+ cluster environment and cluster weather (main model)"]
for label in list(runs)[:6]:  # same 2022 children and outcome as the main model: paired difference
    if "main model)" not in label:
        rows[label]["auc_diff_vs_main"], rows[label]["diff_ci_low"], rows[label]["diff_ci_high"] = paired_diff(sens_preds[label], p_main)
sensitivity = pd.DataFrame(rows).T.astype({"children": int, "positive": int})
sensitivity.round(3)

## Save results

In [ ]:
with pd.ExcelWriter(OUT / "results.xlsx") as xw:
    cv_table.to_excel(xw, sheet_name="Cross-validation 2019", index=False)
    pd.DataFrame(params).T.to_excel(xw, sheet_name="Hyperparameters")
    final_table.to_frame("value").to_excel(xw, sheet_name="Final test 2022")
    weighted.to_frame("value").to_excel(xw, sheet_name="Weighted (selected model)")
    test_table.to_excel(xw, sheet_name="All models 2022", index=False)
    auc_table.to_excel(xw, sheet_name="Train-CV-Test AUC", index=False)
    delong_table.to_excel(xw, sheet_name="DeLong", index=False)
    baseline_table.to_excel(xw, sheet_name="DeLong vs LR", index=False)
    cluster_table.to_excel(xw, sheet_name="Cluster bootstrap AUC diff", index=False)
    robust_table.to_excel(xw, sheet_name="Community by cluster size", index=False)
    subgroup_table.to_excel(xw, sheet_name="Subgroups", index=False)
    miss.to_excel(xw, sheet_name="Missing values")
    community_table.to_excel(xw, sheet_name="Community threshold", index=False)
    c22.to_excel(xw, sheet_name="Communities 2022")
    region_table.to_excel(xw, sheet_name="Regions 2022")
    rf_importance.sort_values(ascending=False).rename(LABELS).to_frame("importance").to_excel(xw, sheet_name="RF importance")
    perm.sort_values("auc_drop", ascending=False).to_excel(xw, sheet_name="Permutation importance", index=False)
    sensitivity.to_excel(xw, sheet_name="Sensitivity")
    positivity.to_excel(xw, sheet_name="Weighted positivity by survey")
print("Saved to", OUT)